# 02 - Kinematics And Coordinate Frames

## What / Why / How

**What we are trying to do:** Learn how robots describe where things are using coordinate frames, transforms, forward kinematics, inverse kinematics, and Jacobians.

**Why this matters:** Robots fail when frames are confused. Cameras, arms, bases, grippers, and maps all need explicit frame relationships.

**How we will do it:** Start with 2D transforms, compute a 2-link arm endpoint, solve for joint angles, and use the Jacobian to connect joint motion to endpoint motion.

## Goal

Learn how robots represent geometry:

- Coordinate frames
- Rotation matrices
- Homogeneous transforms
- Forward and inverse kinematics
- Jacobians

This is the language used by arms, mobile robots, cameras, grippers, and simulation engines.

In [ ]:
import math
import random
from collections import defaultdict

import numpy as np

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see the plot: pip install -r requirements.txt")

## 2D Rotation And Transform Matrices

In [ ]:
def rot2(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])

def transform2(theta, tx, ty):
    T = np.eye(3)
    T[:2, :2] = rot2(theta)
    T[:2, 2] = [tx, ty]
    return T

def apply_transform(T, point_xy):
    p = np.array([point_xy[0], point_xy[1], 1.0])
    return (T @ p)[:2]

T_world_robot = transform2(theta=np.deg2rad(30), tx=1.0, ty=0.5)
point_robot = np.array([0.4, 0.0])
point_world = apply_transform(T_world_robot, point_robot)
print("T_world_robot:")
print(T_world_robot)
print("point in world:", point_world)

## Forward Kinematics: 2-Link Arm

In [ ]:
def fk_2link(q, lengths=(1.0, 0.7)):
    q1, q2 = q
    l1, l2 = lengths
    elbow = np.array([l1 * np.cos(q1), l1 * np.sin(q1)])
    wrist = elbow + np.array([l2 * np.cos(q1 + q2), l2 * np.sin(q1 + q2)])
    return elbow, wrist

q = np.deg2rad([35, 55])
elbow, wrist = fk_2link(q)
print("elbow:", elbow)
print("end effector:", wrist)

In [ ]:
if HAS_PLOT:
    plt.figure(figsize=(4, 4))
    xs = [0, elbow[0], wrist[0]]
    ys = [0, elbow[1], wrist[1]]
    plt.plot(xs, ys, marker="o", linewidth=3)
    plt.axis("equal")
    plt.xlim(-1.8, 1.8)
    plt.ylim(-1.8, 1.8)
    plt.grid(True, alpha=0.3)
    plt.title("2-link arm")
    plt.show()
else:
    plot_unavailable()

## Inverse Kinematics

In [ ]:
def ik_2link(target_xy, lengths=(1.0, 0.7), elbow_up=True):
    x, y = target_xy
    l1, l2 = lengths
    r2 = x * x + y * y
    c2 = (r2 - l1 * l1 - l2 * l2) / (2 * l1 * l2)
    c2 = np.clip(c2, -1.0, 1.0)
    s2 = np.sqrt(max(0.0, 1 - c2 * c2))
    if not elbow_up:
        s2 = -s2
    q2 = np.arctan2(s2, c2)
    q1 = np.arctan2(y, x) - np.arctan2(l2 * s2, l1 + l2 * c2)
    return np.array([q1, q2])

target_xy = np.array([1.0, 0.8])
q_sol = ik_2link(target_xy, elbow_up=True)
_, reached = fk_2link(q_sol)
print("q degrees:", np.rad2deg(q_sol))
print("target:", target_xy, "reached:", reached, "error:", np.linalg.norm(target_xy - reached))

## Jacobian: Joint Velocity To End-Effector Velocity

In [ ]:
def jacobian_2link(q, lengths=(1.0, 0.7)):
    q1, q2 = q
    l1, l2 = lengths
    return np.array([
        [-l1*np.sin(q1) - l2*np.sin(q1+q2), -l2*np.sin(q1+q2)],
        [ l1*np.cos(q1) + l2*np.cos(q1+q2),  l2*np.cos(q1+q2)],
    ])

J = jacobian_2link(q_sol)
qdot = np.array([0.2, -0.1])
ee_velocity = J @ qdot
print("Jacobian:")
print(J)
print("end-effector velocity:", ee_velocity)

## Exercises

1. Try targets outside the arm reach. What does clipping do?
2. Compare elbow-up and elbow-down inverse kinematics.
3. Use the Jacobian pseudo-inverse to move the endpoint toward a target.